# 01 · Robust shift and a sample-aware state report

**ScGeo 0.2.0 learning track.** Start with 720 synthetic cells, three states and four samples per condition. The coordinates are not fitted expression or observed biology.

Use the [separate tutorial environment](README.md). Run cells in order in a fresh kernel. Outputs are written to a new ignored results directory; manuscript inputs and outputs are not used.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import scgeo as sg
from IPython.display import display

cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / "scripts/scgeo_v020_examples/SOURCE_PART1.json").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the scgeo-notebooks checkout.")
examples = root / "scripts/scgeo_v020_examples"
if str(examples) not in sys.path:
    sys.path.insert(0, str(examples))  # only tutorial helpers, not package source
from tutorial_support import setup, save_preview, show_gallery, FORMATS
out = setup(root, "01_robust_shift_report")

## AnnData and the contrast

The input representation is X_toy (three quantitative coordinates). X is only a placeholder here.

In [ ]:
from quickstart import make_toy_data
adata = make_toy_data(seed=7)
display(adata.obs.head())
display(adata.obs.groupby(["condition", "sample", "cell_type"], observed=True).size().rename("cells").reset_index().head(12))

## Run ScGeo, not a hand-coded pseudobulk approximation

Use sample-level aggregation and resampling with a geometric median. State-level effect magnitude and uncertainty are distinct from composition and dynamics.

In [ ]:
sg.tl.robust_shift(adata, rep="X_toy", condition_key="condition",
                   group0="control", group1="treated", by="cell_type",
                   sample_key="sample", center="geometric_median", n_boot=100, seed=7)
report = sg.get.state_report(adata, node_key="cell_type")
display(report[["state", "delta_norm", "n_samples0", "n_samples1", "bootstrap_unit"]])
report.to_csv(out / "state_report.csv", index=False)
assert set(report["bootstrap_unit"]) == {"sample"}

In [ ]:
with sg.pl.style_context("paper"):
    result = sg.pl.state_evidence_panel(report, effect_metric="delta_norm",
        show_sample_counts=True, show_inference=True, return_result=True, show=False)
    save_preview(result, out, "state_evidence")
    samples = sg.get.sample_centers(adata)
    display(samples.head())
    samples.to_csv(out / "sample_centers.csv", index=False)
    result = sg.pl.replicate_effect(samples, coordinate=0,
        condition_order=["control", "treated"], return_result=True, show=False)
    save_preview(result, out, "sample_effects")

**Interpretation:** one point in the replicate plot is one biological sample. The coordinate axis is not a time axis. No representation-stability, local-geometry or dynamics evidence is implied unless separately computed.